# Sıfırdan Multi-Head Attention ve Transformer Block

Bu notebook, araştırma dosyalarında (`06_nlp_ve_transformer_temelleri.md`) anlatılan
Transformer bileşenlerini — **Positional Encoding, Query-Key-Value Attention, Multi-Head
Attention, Transformer Block (Layer Norm + Feed Forward), Token Prediction** — PyTorch ile
sıfırdan (HuggingFace/hazır kütüphane kullanmadan) inşa ediyor.

Amaç kavramları ezbere bilmek değil, her adımın tensör boyutlarını gerçekten görüp
"bu sayı neden bu şekle dönüştü" sorusuna cevap verebilmek — tıpkı daha önceki
`tensor_reshape.ipynb` alıştırmasında yaptığımız gibi. Her hücreden sonra `print(x.shape)`
ile takip ediyoruz.

In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)  # tekrarlanabilir sonuçlar için

## 1) Toy Girdi Hazırlığı

Gerçek bir tokenizer yerine, anlaşılması kolay olsun diye 9 kelimelik minik bir sözlük
kullanıyoruz. 2 cümlelik bir batch, her cümle 4 token (kısa cümleler `<pad>` ile 4'e
tamamlanmış).

In [2]:
VOCAB = ["<pad>", "ben", "kitabı", "okudum", "sen", "kalemi", "aldın", "o", "yazdı"]
vocab_size = len(VOCAB)

d_model = 32       # embedding boyutu
seq_len = 4        # cümle uzunluğu (token sayısı)
batch_size = 2     # kaç cümle
num_heads = 4       # attention head sayısı

cumle1 = ["ben", "kitabı", "okudum", "<pad>"]
cumle2 = ["sen", "kalemi", "aldın", "<pad>"]

token_ids = torch.tensor([
    [VOCAB.index(t) for t in cumle1],
    [VOCAB.index(t) for t in cumle2],
])
print("token_ids.shape:", token_ids.shape)
print(token_ids)

token_ids.shape: torch.Size([2, 4])
tensor([[1, 2, 3, 0],
        [4, 5, 6, 0]])


`token_ids.shape` = `(batch=2, seq_len=4)` — her sayı, VOCAB listesindeki bir kelimenin
index'i. Bu, `tensor_reshape.ipynb`'deki `(32, 10, 64)` örneğinin "embedding'den önceki"
hali gibi düşünülebilir; henüz sayısal bir anlam vektörü değil, sadece kelime ID'leri.

## 2) Embedding

Her kelime ID'sini `d_model=32` boyutlu, öğrenilebilir bir vektöre çeviriyoruz
(bkz. `06_nlp_ve_transformer_temelleri.md` — Embedding bölümü).

In [3]:
embedding = nn.Embedding(vocab_size, d_model)
x = embedding(token_ids)
print("embedding sonrası x.shape:", x.shape)

embedding sonrası x.shape: torch.Size([2, 4, 32])


`(2, 4, 32)` — artık her token, 32 sayıdan oluşan bir vektörle temsil ediliyor. Bu tam olarak
`tensor_reshape.ipynb`'deki `(batch, sequence, embedding)` yapısının aynısı.

## 3) Positional Encoding

Attention, tüm token'ları paralel işlediği için sıra bilgisini kendiliğinden bilmez
(bkz. `06_nlp_ve_transformer_temelleri.md` — Positional Encoding bölümü). Orijinal
"Attention Is All You Need" makalesindeki sabit sinüzoidal formülü kullanıyoruz:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}}) \qquad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [4]:
def positional_encoding(seq_len: int, d_model: int) -> torch.Tensor:
    """Sabit (öğrenilmeyen) sinüzoidal pozisyon kodlamasını üretir."""
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


pe = positional_encoding(seq_len, d_model)
print("positional_encoding.shape:", pe.shape)

x = x + pe  # broadcasting: (2,4,32) + (4,32) -> (2,4,32)
print("pozisyon eklendikten sonra x.shape:", x.shape)

positional_encoding.shape: torch.Size([4, 32])
pozisyon eklendikten sonra x.shape: torch.Size([2, 4, 32])


`pe.shape = (4, 32)` — her pozisyon (0,1,2,3) için 32 boyutlu bir vektör. `x + pe` işleminde
PyTorch'un **broadcasting** özelliği devreye giriyor: `(2,4,32)` boyutlu `x` ile `(4,32)`
boyutlu `pe`, boyutlar uyumlu olduğu için otomatik olarak hizalanıp toplanabiliyor — `pe`,
batch'teki her iki cümleye de aynı şekilde eklenir. Sonuç yine `(2,4,32)`.

## 4) Scaled Dot-Product Attention (Tek Head)

Önce mekanizmayı tek bir head ile, elle kuruyoruz — sonra bunu Multi-Head'e genişleteceğiz.
Her token'ın embedding'i üç ayrı öğrenilmiş `Linear` katmanından geçirilerek Query, Key,
Value vektörlerine dönüştürülür (bkz. `06_nlp_ve_transformer_temelleri.md` — Q/K/V bölümü).

In [5]:
def scaled_dot_product_attention(Q, K, V):
    """Attention(Q,K,V) = softmax(Q @ K^T / sqrt(d_k)) @ V"""
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


Wq = nn.Linear(d_model, d_model)
Wk = nn.Linear(d_model, d_model)
Wv = nn.Linear(d_model, d_model)

Q = Wq(x)
K = Wk(x)
V = Wv(x)
print("Q.shape:", Q.shape, "| K.shape:", K.shape, "| V.shape:", V.shape)

attn_out, attn_weights = scaled_dot_product_attention(Q, K, V)
print("tek-head attention çıktı.shape:", attn_out.shape)
print("attn_weights.shape:", attn_weights.shape)

Q.shape: torch.Size([2, 4, 32]) | K.shape: torch.Size([2, 4, 32]) | V.shape: torch.Size([2, 4, 32])
tek-head attention çıktı.shape: torch.Size([2, 4, 32])
attn_weights.shape: torch.Size([2, 4, 4])


`attn_weights.shape = (2, 4, 4)` — her cümle için 4×4'lük bir matris: satır *i*, sütun *j*
değeri "i. token'ın j. token'a ne kadar dikkat ettiğini" gösterir (her satır softmax'tan
geçtiği için toplamı 1'dir). Aşağıda ilk cümlenin bu matrisine bakalım:

In [6]:
print(f"{'':>10}", *[f"{t:>10}" for t in cumle1])
for i, satir in enumerate(attn_weights[0].detach()):
    degerler = " ".join(f"{v:>10.3f}" for v in satir)
    print(f"{cumle1[i]:>10} {degerler}")

                  ben     kitabı     okudum      <pad>
       ben      0.194      0.226      0.348      0.233
    kitabı      0.242      0.218      0.418      0.122
    okudum      0.167      0.247      0.344      0.242
     <pad>      0.216      0.254      0.323      0.207


Model henüz hiç eğitilmedi (weight'ler rastgele başlatıldı), bu yüzden bu dağılım anlamlı
bir dilbilgisel ilişkiyi yansıtmıyor — sadece mekanizmanın matematiksel olarak **çalıştığını**
gösteriyor. Eğitim sonrasında (ör. gerçek bir dil modelinde) bu matrisin, birbiriyle anlamca
ilişkili token'lar arasında belirgin şekilde yüksek değerler alması beklenir.

## 5) Multi-Head Attention

Tek head yerine, aynı işlemi `num_heads=4` farklı, birbirinden bağımsız öğrenilmiş
Q/K/V projeksiyonuyla paralel yapıyoruz (bkz. `06_nlp_ve_transformer_temelleri.md` —
Multi-Head Attention bölümü). `d_model=32` boyutu 4 head'e bölünür, her head
`d_k = 32/4 = 8` boyutunda çalışır; sonunda tüm head'lerin çıktıları birleştirilip (concat)
tek bir `Linear` (`Wo`) ile tekrar `d_model` boyutuna indirgenir.

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model, num_heads'e tam bölünmeli"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def split_heads(self, t: torch.Tensor) -> torch.Tensor:
        # (batch, seq, d_model) -> (batch, seq, num_heads, d_k) -> (batch, num_heads, seq, d_k)
        batch, seq, d_model = t.shape
        t = t.view(batch, seq, self.num_heads, self.d_k)
        return t.transpose(1, 2)

    def forward(self, x: torch.Tensor):
        Q = self.split_heads(self.Wq(x))
        K = self.split_heads(self.Wk(x))
        V = self.split_heads(self.Wv(x))

        d_k = Q.shape[-1]
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
        weights = torch.softmax(scores, dim=-1)
        heads_out = weights @ V

        # Head'leri birleştir: (batch, num_heads, seq, d_k) -> (batch, seq, d_model)
        batch, num_heads, seq, d_k = heads_out.shape
        heads_out = heads_out.transpose(1, 2).contiguous().view(batch, seq, num_heads * d_k)

        return self.Wo(heads_out), weights


mha = MultiHeadAttention(d_model, num_heads)
mha_out, mha_weights = mha(x)
print("MultiHeadAttention çıktı.shape:", mha_out.shape)
print("MultiHeadAttention attn_weights.shape:", mha_weights.shape)

MultiHeadAttention çıktı.shape: torch.Size([2, 4, 32])
MultiHeadAttention attn_weights.shape: torch.Size([2, 4, 4, 4])


`mha_weights.shape = (2, 4, 4, 4)` = `(batch, num_heads, seq, seq)` — 4 head'in her biri
kendi 4×4 attention matrisini üretiyor. Çıktı (`mha_out`) ise tekrar `(2, 4, 32)` — yani
Multi-Head Attention, boyutları değiştirmeden (girdiyle aynı shape'te) çalışır; bu, onu bir
Transformer bloğunun içine (residual connection ile) sorunsuz yerleştirmemizi sağlar.

`split_heads` içindeki `.view()` çağrısının çalışması için `x`'in **contiguous** olması
gerekiyordu (bkz. `tensor_reshape.ipynb`), bu yüzden birleştirme adımında `.transpose()`
sonrası `.contiguous()` çağrısını atlamadık — atlarsaydık `RuntimeError` alırdık.

## 6) Transformer Block

Multi-Head Attention'ın etrafına Residual (Add) bağlantıları, Layer Normalization ve bir
Feed Forward ağı ekleyerek tam bir Transformer bloğu kuruyoruz
(bkz. `06_nlp_ve_transformer_temelleri.md` — Transformer Block bölümü):

```
x -> Multi-Head Attention -> Add(x) -> LayerNorm -> Feed Forward -> Add -> LayerNorm -> çıktı
```

In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.mha(x)
        x = self.norm1(x + attn_out)     # Residual + LayerNorm
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)       # Residual + LayerNorm
        return x


block = TransformerBlock(d_model, num_heads, d_ff=64)
block_out = block(x)
print("TransformerBlock çıktı.shape:", block_out.shape)

TransformerBlock çıktı.shape: torch.Size([2, 4, 32])


Bir bloktan geçtikten sonra bile shape hâlâ `(2, 4, 32)` — bu **kasıtlı bir tasarım**: bir
Transformer bloğu girdisiyle aynı boyutta çıktı ürettiği için, bu bloklardan istediğimiz kadar
üst üste dizebiliriz (gerçek modellerde onlarca/yüzlerce tane). Aşağıda 2 bloğu art arda
bağlıyoruz:

In [9]:
N = 2  # blok sayısı
blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff=64) for _ in range(N)])

h = x
for i, b in enumerate(blocks, start=1):
    h = b(h)
    print(f"blok {i} sonrası h.shape:", h.shape)

blok 1 sonrası h.shape: torch.Size([2, 4, 32])
blok 2 sonrası h.shape: torch.Size([2, 4, 32])


## 7) Token Prediction (Next Token Prediction)

Son bloğun çıktısını bir `Linear` katmanla sözlük boyutuna (`vocab_size=9`) genişletip
softmax uyguluyoruz — her token pozisyonu için, sözlükteki her kelimenin "bir sonraki
kelime olma olasılığını" üretiyoruz (bkz. `06_nlp_ve_transformer_temelleri.md` —
Token Prediction bölümü).

In [10]:
lm_head = nn.Linear(d_model, vocab_size)
logits = lm_head(h)
print("logits.shape:", logits.shape)

probs = torch.softmax(logits, dim=-1)
tahmin_id = probs.argmax(dim=-1)
print("tahmin_id.shape:", tahmin_id.shape)

for cumle_no, cumle in enumerate([cumle1, cumle2]):
    tahminler = [VOCAB[i] for i in tahmin_id[cumle_no].tolist()]
    print(f"Girdi:  {cumle}")
    print(f"Tahmin: {tahminler}\n")

logits.shape: torch.Size([2, 4, 9])
tahmin_id.shape: torch.Size([2, 4])
Girdi:  ['ben', 'kitabı', 'okudum', '<pad>']
Tahmin: ['kitabı', 'yazdı', 'kitabı', 'kitabı']

Girdi:  ['sen', 'kalemi', 'aldın', '<pad>']
Tahmin: ['sen', 'yazdı', 'kitabı', 'kalemi']



`logits.shape = (2, 4, 9)` = `(batch, seq_len, vocab_size)` — her token pozisyonu için
9 kelimelik sözlükteki her kelimeye bir skor. Tahminler anlamsız çıkıyor çünkü model
**hiç eğitilmedi** — tüm `Linear` katmanları rastgele başlatılmış ağırlıklarla çalışıyor.

Gerçek bir dil modeli, tam olarak bu iskelet üzerinde, milyarlarca gerçek cümleyle,
`Backpropagation` + `Optimizer` (bkz. `04_deep_learning_temelleri.md`) kullanılarak
eğitilir — `lm_head`'in ürettiği olasılıklar gerçek dilin istatistiklerine (bkz.
`01_istatistik_temelleri.md`) yaklaşana kadar bu döngü tekrarlanır.

## Özet

Bu notebook'ta uçtan uca kurduğumuz veri akışı:

```
token_ids (2,4)
  -> Embedding -> (2,4,32)
  -> + Positional Encoding -> (2,4,32)
  -> N × TransformerBlock (Multi-Head Attention + Add&Norm + FeedForward + Add&Norm) -> (2,4,32)
  -> lm_head (Linear) -> logits (2,4,9)
  -> softmax + argmax -> tahmin edilen token'lar
```

Bu, GPT tarzı (decoder-only) bir Transformer'ın temel iskeletinin ta kendisi — gerçek
modeller sadece `d_model`, `num_heads`, blok sayısı (`N`) ve `vocab_size` değerlerini çok
daha büyütüyor (ör. GPT-3: `d_model=12288`, 96 blok, ~50.000 kelimelik sözlük) ve devasa
miktarda veriyle eğitiyor.